In [1]:
!pip install tensorflow==2.19.0

In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import os
import cv2
import numpy as np
import albumentations as A
import random
import tensorflow_hub as hub
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2B0
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing.image import save_img
from PIL import Image
# from keras.layers import TFSMLayer
# from keras import Sequential


2026-05-19 09:30:49.359384: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779183049.560436      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779183049.618900      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779183050.089533      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779183050.089570      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779183050.089573      57 computation_placer.cc:177] computation placer alr

In [3]:
#Thông số các biến
image_shape = (32,32)
# Tạo ánh xạ nhãn theo thứ tự mong muốn
label_map = {
    "nấm mỡ": 0,
    "bào ngư xám + trắng": 1,
    "Đùi gà Baby (cắt ngắn)": 2,
    "linh chi trắng": 3
}

In [5]:
!git clone https://github.com/ChipVi/NAM.git

fatal: destination path 'NAM' already exists and is not an empty directory.


In [7]:
%cd /kaggle/working/NAM
!ls

/kaggle/working/NAM
cnn-model.ipynb  simple_CNN_Vi.ipynb  train	     val_augment
garbage1.ipynb	 submission.csv       train_augment
inference.ipynb  test		      val


In [12]:
!ls

cnn-model.ipynb  simple_CNN_Vi.ipynb  train	     val_augment
garbage1.ipynb	 submission.csv       train_augment
inference.ipynb  test		      val


In [17]:
#Đọc file ảnh từ thư mục
input_test = r"/kaggle/working/NAM/test"
output_test = r"/kaggle/working/NAM/submission.csv"

input_train = r"/kaggle/working/NAM/train"
train_augment = r"/kaggle/working/NAM/train_augment"
input_val = r"/kaggle/working/NAM/val";
val_augment = r"/kaggle/working/NAM/val_augment"

train_splitted = r"/kaggle/working/NAM/train_splitted"

if not os.path.exists(train_splitted):
    os.makedirs(train_splitted)
if not os.path.exists(input_val):
    os.makedirs(input_val)
if not os.path.exists(train_augment):
    os.makedirs(train_augment)
if not os.path.exists(input_train):
    os.makedirs(input_train)
if not os.path.exists(val_augment):
    os.makedirs(val_augment)

In [19]:
split_train_folder(
    train_folder="/kaggle/working/NAM/train",
    output_folder="/kaggle/working/NAM/train_splitted",
    train_ratio=0.8,
    seed = 77
)

bào ngư xám + trắng: 200 train_after | 50 valid
nấm mỡ: 200 train_after | 50 valid
Đùi gà Baby (cắt ngắn): 200 train_after | 50 valid
linh chi trắng: 200 train_after | 50 valid


In [ ]:
# AUGMENT FILES PHYSICALLY
def augment_image(input_folder, output_folder):
    transform = A.Compose([
        A.OneOf([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5)
        ], p=0.8),

        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.8),

        A.OneOf([
            A.GaussianBlur(blur_limit=3, p=0.5),
        ], p=0.5),

        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.8),

        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.5)
    ])

    # For each class
    for class_name in os.listdir(input_folder):
        class_path = os.path.join(input_folder, class_name)
        if not os.path.isdir(class_path):
            continue

        output_class_path = os.path.join(output_folder, class_name)
        os.makedirs(output_class_path, exist_ok=True)

        for image_name in os.listdir(class_path):
            image_path = os.path.join(class_path, image_name)

            # Load image safely (unicode support)
            try:
                pil_image = Image.open(image_path).convert("RGB")
            except Exception as e:
                print(f"Cannot open image: {image_path}, error: {e}")
                continue

            image = np.array(pil_image)

            # Save original image
            save_path_original = os.path.join(output_class_path, f"orig_{image_name}")
            save_img(save_path_original, image)

            #  5 transforms cố định
            for i in range(5):
                augmented = transform(image=image)
                aug_image = augmented["image"]
                save_path = os.path.join(output_class_path, f"aug{i+1}_{image_name}")
                save_img(save_path, aug_image)
                print(f"Saved: {save_path}")


# Augmentation data training and validation
augment_image('/kaggle/working/NAM/train_splitted/train_after', train_augment)
augment_image('/kaggle/working/NAM/train_splitted/valid', val_augment)


In [68]:
import os

folder = r'/kaggle/working/NAM/train_augment/Đùi gà Baby (cắt ngắn)'

count = len(os.listdir(folder))

print(count)

1500


In [71]:
datagen = ImageDataGenerator(
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3]
)

batch_size = 128
target_size= (64,64)

# Load dữ liệu train
train_generator = datagen.flow_from_directory(
    train_augment,
    target_size= target_size,
    batch_size=batch_size,
    class_mode='sparse',
    shuffle=True
)

# Load dữ liệu val
val_generator = datagen.flow_from_directory(
    val_augment,
    target_size= target_size,
    batch_size=batch_size,
    class_mode='sparse',
    shuffle=True
)

# Load tập test
test_generator = datagen.flow_from_directory(
    input_test,
    target_size= target_size,
    batch_size=1,
    class_mode='sparse',
    shuffle=True
)

# Cập nhật lại nhãn theo ánh xạ tùy chỉnh
# Lấy tên class từ đường dẫn file
train_generator.class_indices = label_map
train_generator.classes = np.array([
    label_map[os.path.dirname(file)] for file in train_generator.filenames
])

val_generator.class_indices = label_map
val_generator.classes = np.array([
    label_map[os.path.dirname(file)] for file in val_generator.filenames
])

# Kiểm tra xem nhãn đã đúng chưa
print(train_generator.class_indices)
print(val_generator.class_indices)

Found 6000 images belonging to 4 classes.
Found 2400 images belonging to 4 classes.
Found 200 images belonging to 4 classes.
{'nấm mỡ': 0, 'bào ngư xám + trắng': 1, 'Đùi gà Baby (cắt ngắn)': 2, 'linh chi trắng': 3}
{'nấm mỡ': 0, 'bào ngư xám + trắng': 1, 'Đùi gà Baby (cắt ngắn)': 2, 'linh chi trắng': 3}


In [72]:
x, y = next(train_generator)

print(x.shape)
print(y.shape)

(128, 64, 64, 3)
(128,)


In [44]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(64,64,3)
)

In [73]:
# freeze pretrained layers
base_model.trainable = False

NUM_CLASSES=4
model = models.Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.6),

    layers.Dense(NUM_CLASSES, activation='softmax')
])

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# =========================
# TRAIN
# =========================
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20
)

Epoch 1/20


2026-05-19 12:10:24.702030: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:10:24.839279: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:10:25.182902: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:10:25.327965: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:10:26.252378: E external/local_xla/xla/stream_

21/47 ━━━━━━━━━━━━━━━━━━━━ 6s 232ms/step - accuracy: 0.3774 - loss: 1.3319

2026-05-19 12:10:45.887326: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:10:46.024035: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:10:46.357668: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:10:46.499624: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:10:47.368241: E external/local_xla/xla/stream_

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 595ms/step - accuracy: 0.4675 - loss: 1.1826

2026-05-19 12:11:18.089550: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:11:18.225592: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:11:18.567589: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:11:18.709750: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 12:11:18.851021: E external/local_xla/xla/stream_

47/47 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.4699 - loss: 1.1784 - val_accuracy: 0.7221 - val_loss: 0.7456
Epoch 2/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 349ms/step - accuracy: 0.7297 - loss: 0.6982 - val_accuracy: 0.7546 - val_loss: 0.6528
Epoch 3/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 345ms/step - accuracy: 0.7527 - loss: 0.6398 - val_accuracy: 0.7575 - val_loss: 0.6246
Epoch 4/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 350ms/step - accuracy: 0.7806 - loss: 0.5825 - val_accuracy: 0.7733 - val_loss: 0.6097
Epoch 5/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 348ms/step - accuracy: 0.7910 - loss: 0.5606 - val_accuracy: 0.7858 - val_loss: 0.5735
Epoch 6/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 353ms/step - accuracy: 0.8070 - loss: 0.5143 - val_accuracy: 0.7833 - val_loss: 0.5585
Epoch 7/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 349ms/step - accuracy: 0.8151 - loss: 0.4947 - val_accuracy: 0.7867 - val_loss: 0.5502
Epoch 8/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 343ms/step - accuracy: 0.8131 - loss: 0.4923 - val_accuracy: 0.7933 -

In [74]:
test_loss, test_acc = model.evaluate(test_generator)

print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# ====================================
# PREDICT
# ====================================
pred_probs = model.predict(test_generator)

pred_classes = np.argmax(pred_probs, axis=1)

true_classes = test_generator.classes

class_names = list(test_generator.class_indices.keys())

# ====================================
# CLASSIFICATION REPORT
# ====================================
print("\nClassification Report:\n")

print(classification_report(
    true_classes,
    pred_classes,
    target_names=class_names
))



200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.0955 - loss: 5.8966

Test Accuracy: 0.0950
Test Loss: 5.7742
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step

Classification Report:

              precision    recall  f1-score   support

     Bào ngư       0.09      0.10      0.09        50
    Linh chi       0.22      0.22      0.22        50
      Nấm mỡ       0.29      0.14      0.19        50
      Đùi gà       0.23      0.32      0.27        50

    accuracy                           0.20       200
   macro avg       0.21      0.20      0.19       200
weighted avg       0.21      0.20      0.19       200

